# AeroTest: Statistical Analysis & A/B Testing Framework
### Domain: Product Analytics & Growth Data Science

This notebook demonstrates a rigorous, production-grade statistical analysis pipeline for A/B testing. It covers:
1. **Synthetic Data Generation**: Simulating user-level transaction and conversion data.
2. **Pre-analysis checks**: Sample Ratio Mismatch (SRM) detection via Chi-Square Goodness-of-Fit.
3. **Hypothesis Testing**: Z-test for conversion rates (proportions) and Student's t-test for Average Revenue Per User (continuous).
4. **Bootstrapping**: Non-parametric confidence intervals for robust business decision-making.
5. **Power Analysis**: Post-hoc power analysis and minimum detectable effect (MDE) curves.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import TTestIndPower

# Set style for visualizations
sns.set_theme(style="whitegrid")
np.random.seed(42)
print("Libraries successfully imported!")

## 1. Synthetic Data Generation
We will simulate user engagement data for an e-commerce platform testing a new checkout page design:
- **Control (A)**: Current layout (expected conversion rate ~ 12%)
- **Treatment (B)**: Sleek layout (expected conversion rate ~ 13.5%)
- **Target Sample Size**: 10,000 users per group

In [ ]:
n_control = 10050  # slightly uneven to simulate real-world noise
n_treatment = 9950

# Simulate conversion (binomial)
p_control_true = 0.12
p_treatment_true = 0.135

conv_control = np.random.binomial(1, p_control_true, n_control)
conv_treatment = np.random.binomial(1, p_treatment_true, n_treatment)

# Simulate Revenue per converting customer (log-normal distribution)
rev_control = np.where(conv_control == 1, np.random.lognormal(mean=3.5, sigma=0.5, size=n_control), 0.0)
rev_treatment = np.where(conv_treatment == 1, np.random.lognormal(mean=3.6, sigma=0.45, size=n_treatment), 0.0)

# Create DataFrame
df_control = pd.DataFrame({
    'user_id': range(1, n_control + 1),
    'group': 'control',
    'converted': conv_control,
    'revenue': rev_control
})

df_treatment = pd.DataFrame({
    'user_id': range(n_control + 1, n_control + n_treatment + 1),
    'group': 'treatment',
    'converted': conv_treatment,
    'revenue': rev_treatment
})

df = pd.concat([df_control, df_treatment], ignore_index=True)
df.head()

## 2. Sample Ratio Mismatch (SRM) Check
Sample Ratio Mismatch is a critical data quality check. If the actual sample split deviates significantly from the expected ratio (e.g., 50/50), it indicates a bug in the routing mechanism, rendering downstream statistical tests invalid.

We use a Chi-Square Goodness-of-Fit test.

In [ ]:
observed = [n_control, n_treatment]
total = sum(observed)
expected = [total / 2, total / 2]  # Assuming a 50/50 split target

chi2_stat, p_val_srm = stats.chisquare(f_obs=observed, f_exp=expected)
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"SRM Check p-value: {p_val_srm:.4f}")

if p_val_srm < 0.001:
    print("WARNING: Potential Sample Ratio Mismatch (SRM) detected! Investigate randomization pipeline.")
else:
    print("SUCCESS: No SRM detected. Randomization appears valid.")

## 3. Hypothesis Testing: Conversion Rate (Proportions Z-Test)
- $H_0: p_{treatment} = p_{control}$ (The sleek checkout page does not affect conversion rate)
- $H_1: p_{treatment} \neq p_{control}$ (The sleek checkout page does affect conversion rate)
- Significance level $\alpha = 0.05$

In [ ]:
conversions = [df_control['converted'].sum(), df_treatment['converted'].sum()]
nobs = [n_control, n_treatment]

z_stat, p_val_z = proportions_ztest(conversions, nobs, alternative='two-sided')
print(f"Z-statistic: {z_stat:.4f}")
print(f"p-value: {p_val_z:.4f}")

cr_control = conversions[0] / nobs[0]
cr_treatment = conversions[1] / nobs[1]
uplift = (cr_treatment - cr_control) / cr_control * 100

print(f"Control Conversion Rate: {cr_control:.4%}")
print(f"Treatment Conversion Rate: {cr_treatment:.4%}")
print(f"Relative Conversion Uplift: {uplift:.2f}%")

if p_val_z < 0.05:
    print("Decision: Reject H0. The checkout page change has a statistically significant effect.")
else:
    print("Decision: Fail to reject H0. No statistically significant change.")

## 4. Hypothesis Testing: Average Revenue Per User (T-Test)
Since revenue per user is highly skewed (lots of zeros and a log-normal tail for buyers), standard t-tests are robust at large sample sizes due to the Central Limit Theorem. However, we will also use a T-test here to analyze the mean differences.

In [ ]:
t_stat, p_val_t = stats.ttest_ind(df_control['revenue'], df_treatment['revenue'], equal_var=False)
print(f"T-statistic: {t_stat:.4f}")
print(f"p-value: {p_val_t:.4f}")

arpu_control = df_control['revenue'].mean()
arpu_treatment = df_treatment['revenue'].mean()
print(f"Control ARPU: ${arpu_control:.4f}")
print(f"Treatment ARPU: ${arpu_treatment:.4f}")
print(f"ARPU Lift: ${(arpu_treatment - arpu_control):.4f}")

## 5. Bootstrapping (Non-parametric Confidence Intervals)
To handle the skewed revenue distribution without relying on asymptotic assumptions, we use Bootstrapping to build confidence intervals for the revenue difference.

In [ ]:
n_bootstrap = 2000
bootstrap_diffs = []

control_revenue = df_control['revenue'].values
treatment_revenue = df_treatment['revenue'].values

for _ in range(n_bootstrap):
    boot_control = np.random.choice(control_revenue, size=len(control_revenue), replace=True)
    boot_treatment = np.random.choice(treatment_revenue, size=len(treatment_revenue), replace=True)
    bootstrap_diffs.append(boot_treatment.mean() - boot_control.mean())

bootstrap_diffs = np.array(bootstrap_diffs)
ci_lower = np.percentile(bootstrap_diffs, 2.5)
ci_upper = np.percentile(bootstrap_diffs, 97.5)

print(f"95% Bootstrapped CI for ARPU Difference: [{ci_lower:.4f}, {ci_upper:.4f}]")

plt.figure(figsize=(10, 6))
plt.hist(bootstrap_diffs, bins=40, color='#8884d8', edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', label='No Difference')
plt.axvline(x=ci_lower, color='blue', linestyle=':', label='95% CI Lower')
plt.axvline(x=ci_upper, color='blue', linestyle=':', label='95% CI Upper')
plt.title('Bootstrapped Distribution of ARPU Difference')
plt.xlabel('Difference in Mean Revenue ($)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## 6. Power Analysis Curves
We construct statistical power curves to help key stakeholders understand the sample sizes required to detect various effect sizes in future experiments.

In [ ]:
power_analysis = TTestIndPower()
effect_sizes = [0.02, 0.05, 0.10]  # Cohens d
sample_sizes = np.arange(100, 20000, 100)

plt.figure(figsize=(10, 6))
for es in effect_sizes:
    power_analysis.plot_power(
        dep_var='nobs', 
        nobs=sample_sizes, 
        effect_size=es, 
        alpha=0.05, 
        ax=plt.gca(), 
        title='Power Curves by Effect Size (d)'
)
plt.ylabel('Statistical Power (1 - Beta)')
plt.xlabel('Sample Size (per variant)')
plt.legend([f"d = {es}" for es in effect_sizes])
plt.axhline(y=0.80, color='grey', linestyle='--', label='Target Power (80%)')
plt.show()